# ApproxIoT replication — results analysis

Loads the CSVs produced by `python -m experiments.run_all` from `results/` and
redraws the paper-figure analogues (Figs. 6–12) plus the novel controller
comparison figures. Rerun the experiment suite first if the CSVs are missing.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

RESULTS = Path("results")


def load(name):
    path = RESULTS / name
    return pd.read_csv(path) if path.exists() else None


exp1 = load("exp1_sampling_fraction.csv")
exp2 = load("exp2_window_size.csv")
exp3 = load("exp3_fluctuating_rates.csv")
exp4 = load("exp4_skew.csv")
exp5 = load("exp5_real_world.csv")
exp5_multi = load("exp5_real_world_multiseed.csv")
exp6 = load("exp6_adaptive_controller.csv")
frac6 = load("exp6_fraction_history.csv")

print("available:", {n: df is not None for n, df in [
    ("exp1", exp1), ("exp2", exp2), ("exp3", exp3), ("exp4", exp4),
    ("exp5", exp5), ("exp5_multi", exp5_multi), ("exp6", exp6)]})
if any(df is None for df in [exp1, exp2, exp3, exp4, exp5, exp6]):
    print("run `python -m experiments.run_all` to generate the missing CSVs")

## EXP1 — accuracy loss & throughput vs sampling fraction (Figs. 6–9 analogue)

In [ ]:
if exp1 is not None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex="col")
    for row, dist in enumerate(("gaussian", "poisson")):
        sub = exp1[exp1.dist == dist]
        for method, g in sub[sub.method != "native"].groupby("method"):
            agg = g.groupby("fraction")[["realized_loss", "ingest_throughput_items_per_sec"]].median()
            axes[row, 0].plot(agg.index, agg["realized_loss"], marker="o", label=method)
            axes[row, 1].plot(agg.index, agg["ingest_throughput_items_per_sec"],
                              marker="o", label=method)
        axes[row, 0].set(ylabel="median accuracy loss", yscale="log",
                         title=f"EXP1: accuracy loss ({dist})")
        axes[row, 1].set(ylabel="items processed / simulated second",
                         title=f"EXP1: throughput ({dist})")
        for ax in axes[row]:
            ax.grid(alpha=0.3)
            ax.legend()
    axes[1, 0].set_xlabel("sampling fraction")
    axes[1, 1].set_xlabel("sampling fraction")
    plt.tight_layout()
    plt.show()

## EXP2 — measured latency vs window size (Fig. 10 analogue)

In [ ]:
if exp2 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for method, g in exp2.groupby("method"):
        g = g.sort_values("window_seconds")
        axes[0].plot(g.window_seconds, g.measured_latency_ms, marker="o", label=method)
        axes[1].plot(g.window_seconds, g.throughput, marker="o", label=method)
    axes[0].set(xlabel="window size (seconds)", ylabel="measured e2e latency (ms)",
                title="EXP2: latency (event-trace measurement)")
    axes[1].set(xlabel="window size (seconds)",
                ylabel="items processed / simulated second",
                title="EXP2: simulated throughput (flat vs window)")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()
    display(exp2)

## EXP3 — fluctuating rate settings, Gaussian & Poisson (§V-D / Fig. 11 analogue)

In [ ]:
if exp3 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, dist in zip(axes, ("gaussian", "poisson")):
        piv = exp3[exp3.dist == dist].groupby(["phase", "method"])["realized_loss"].median().unstack("method")
        display(piv)
        piv.plot(marker="o", logy=True, ax=ax, title=f"EXP3: {dist}")
        ax.set(xlabel="rate-setting phase (1→2→3→1)", ylabel="median realized loss")
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## EXP4 — extreme skew (§V-E analogue, the dramatic gap)

In [ ]:
if exp4 is not None:
    piv = exp4.groupby(["fraction", "method"])["realized_loss"].median().unstack("method")
    display(piv)
    ax = piv.clip(lower=1e-8).plot(marker="o", logy=True, figsize=(7, 4),
                                   title="EXP4: skew workload, ApproxIoT vs SRS")
    ax.set(xlabel="sampling fraction", ylabel="median realized loss")
    ax.grid(alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()

## EXP5 — real-world datasets (§VI analogue)

In [ ]:
if exp5 is not None:
    src = exp5_multi if exp5_multi is not None else exp5
    label = "multi-seed" if exp5_multi is not None else "single-seed"
    piv = src.groupby(["fraction", "method"])["realized_loss"].median().unstack("method")
    display(piv)
    ax = piv.clip(lower=1e-8).plot(marker="o", logy=True, figsize=(7, 4),
                                   title=f"EXP5: taxi + air-quality workload ({label})")
    ax.set(xlabel="sampling fraction", ylabel="median realized loss")
    ax.grid(alpha=0.3, which="both")
    plt.tight_layout()
    plt.show()

## EXP6 — the novelty: adaptive controller vs static budget

Top: relative standard error over time for the mid-run rate step-change.
Bottom: the sampling fraction each regime used. Adaptive regimes target 0.5%
(or 0.1% for the `-tight` variants).

In [ ]:
if exp6 is not None:
    step = exp6[(exp6.scenario == "step-change") & (exp6.n_in_root > 0)]
    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    for regime, g in step.groupby("regime"):
        axes[0].plot(g.window, g.stderr_relative, marker=".", ms=4, label=regime)
        axes[1].plot(g.window, g.fraction_root, marker=".", ms=4, label=regime)
    axes[0].axhline(0.005, color="k", ls="--", alpha=0.5, label="target 0.5%")
    axes[0].axvline(20, color="k", ls=":", alpha=0.5)
    axes[1].axvline(20, color="k", ls=":", alpha=0.5)
    axes[0].set(ylabel="relative stderr", title="EXP6: rate step-change — adaptive vs static")
    axes[1].set(xlabel="window", ylabel="sampling fraction (root)")
    for ax in axes:
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize="x-small", ncol=2)
    plt.tight_layout()
    plt.show()

    agg = exp6.groupby(["scenario", "regime"]).agg(
        median_loss=("realized_loss", "median"),
        median_stderr=("stderr_relative", "median"),
        bandwidth_saved=("bandwidth_saved", "mean"),
        violation_rate=("violation", "mean"),
        final_fraction=("fraction_root", "last"),
    )
    display(agg.round(5))

## Verdicts — programmatic qualitative checks

In [ ]:
summary_path = RESULTS / "summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    rows = []
    for name, payload in summary.items():
        check = payload.get("qualitative_check") if isinstance(payload, dict) else None
        if check is not None:
            rows.append({"experiment": name, "pass": check.get("pass"),
                         "detail": {k: v for k, v in check.items() if k not in {"pass", "note"}}})
    display(pd.DataFrame(rows))
else:
    print("run `python -m experiments.run_all` first")